# 07 · Excel 写入模块（Excel）功能演示

演示 ExcelWriter 的表格写入、分箱图、迷你图(sparkline) 与便捷函数 dataframe2excel / DataFrame.save。

In [1]:
import warnings, os
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import hscredit

# 路径约定：从 notebooks/ 目录运行，数据在 ../examples，产物输出到 model_report/
DATA = os.path.join("..", "examples", "hscredit_yyp.xlsx")
if not os.path.exists(DATA):
    DATA = os.path.join("examples", "hscredit_yyp.xlsx")
OUT = "model_report"
os.makedirs(OUT, exist_ok=True)

df = pd.read_excel(DATA)
df["放款时间"] = pd.to_datetime(df["放款时间"])
y = df["FPD"].astype(int)
NUM_FEATURES = ["珊瑚92", "青云24", "衡枢鉴真分老客版", "占信V3", "天创小额网贷分", "近六个月非银多头机构数"]
CAT_FEATURE = "商品类别"
print("数据形状:", df.shape)
print("坏样本率: {:.4f}".format(y.mean()))
df.head()

数据形状: (970, 18)
坏样本率: 0.1402


,客户编号,放款时间,放款金额,商品类别,MOB1,CURRENT_DPD,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,近六个月非银多头机构数,手机号近一个月非银多头机构数,身份证近一个月非银多头机构数,衡枢鉴真分老客版,FPD
0,1985945640026276096,2026-02-03,1399,礼包,0,0,NaN,NaN,NaN,656,NaN,NaN,630,51,15,15,0.0242,0
1,1985972188268592896,2026-02-04,1399,礼包,0,0,NaN,NaN,NaN,565,NaN,NaN,583,56,6,18,0.0492,0
2,1986034700861140992,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,708,NaN,NaN,764,68,17,20,0.0546,0
3,1986264852923760896,2025-11-06,3960,珠宝首饰,0,0,NaN,NaN,NaN,555,NaN,NaN,712,45,15,15,0.0899,0
4,1986265696509906944,2026-01-26,1399,礼包,0,0,NaN,NaN,NaN,581,NaN,NaN,641,67,32,32,0.0678,0


## 1. ExcelWriter：标题 + 分箱表 + 冻结窗格 + 列宽

In [2]:
from hscredit.excel import ExcelWriter, dataframe2excel
from hscredit.core.binning import OptimalBinning

binner = OptimalBinning(method='best_iv').fit(df[['衡枢鉴真分老客版','青云24']], y)
bt = binner.get_bin_table('衡枢鉴真分老客版')

w = ExcelWriter()
ws = w.get_sheet_by_name('分箱表')
next_row, _ = w.insert_value2sheet(ws, (2, 1), value='衡枢鉴真分老客版 · best_iv 分箱明细', style='header')
w.insert_df2sheet(ws, bt, (next_row + 1, 1), header=True)
w.set_column_width(ws, 1, 18)
w.set_freeze_panes(ws, 'A2')
print('已写入分箱表 sheet')

已写入分箱表 sheet


## 2. 分箱图（bin chart）写入单元格

In [3]:
ws2 = w.get_sheet_by_name('分箱图')
w.insert_bin_chart2sheet(ws2, bt, (2, 1))
out1 = f"{OUT}/07_excel_writer_demo.xlsx"
w.save(out1)
print('已保存:', out1)

已保存: model_report/07_excel_writer_demo.xlsx


## 3. 迷你图 sparkline（折线 / 柱状，保存后注入 XML）

In [4]:
w2 = ExcelWriter()
ws = w2.get_sheet_by_name('迷你图')
spark_df = pd.DataFrame(np.random.RandomState(0).rand(5, 6).round(3), columns=[f'M{i+1}' for i in range(6)])
w2.insert_df2sheet(ws, spark_df, (1, 1), header=True)
for r in range(5):
    w2.add_sparkline(ws, f'H{r+2}', f'B{r+2}:G{r+2}', markers=True, high_point=True, low_point=True)
    w2.add_sparkline(ws, f'I{r+2}', f'B{r+2}:G{r+2}', type='column')
out2 = f"{OUT}/07_excel_sparkline_demo.xlsx"
w2.save(out2)

# 校验迷你图 XML 已注入且文件可重新打开
import zipfile
from openpyxl import load_workbook
with zipfile.ZipFile(out2) as z:
    has_spark = 'sparkline' in z.read('xl/worksheets/sheet1.xml').decode('utf-8','ignore').lower()
print('迷你图 XML 已注入:', has_spark, '| 文件可重新打开:', '迷你图' in load_workbook(out2).sheetnames)

迷你图 XML 已注入: True | 文件可重新打开: True


## 4. 便捷函数 dataframe2excel 与 DataFrame.save() 扩展

In [5]:
dataframe2excel(bt, f"{OUT}/07_excel_dataframe2excel.xlsx", sheet_name='IV表', title='IV 分箱表')
bt.save(f"{OUT}/07_excel_df_save.xlsx", sheet_name='数据', title='DataFrame.save 扩展')
y.value_counts().save(f"{OUT}/07_excel_series_save.xlsx", title='Series.save 扩展')
print('已保存 dataframe2excel / DataFrame.save / Series.save 结果')

已保存 dataframe2excel / DataFrame.save / Series.save 结果
